# فصل 14: برنامه‌نویسی Back-End با FastAPI

کد این نوت‌بوک عیناً از فصل 14 کتاب «مهندسی هوش مصنوعی مکانی» برداشته شده است.

پیش از اجرا: مسیر داده‌ها را با ساختار پوشه‌ی `data/`، `work/` و `out/` هماهنگ کنید. راه دانلود هر فایل در پیوست ب کتاب آمده است.


## ۱۴-۲) اولین API با FastAPI


In [ ]:
# terminal:  pip install "fastapi[standard]"
# file: main.py
from fastapi import FastAPI

app = FastAPI(title="GeoAI Flood Service")

@app.get("/")                        # handle GET requests to "/"
def root():
    return {"message": "GeoAI API is running", "docs": "/docs"}

@app.get("/hello/{name}")            # {name} is a path parameter
def hello(name: str):
    return {"greeting": f"Hello, {name}"}

## ۱۴-۳) پارامترها و اعتبارسنجی با Pydantic


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
class HospitalCreate(BaseModel):
    name: str = Field(..., min_length=3, max_length=200) # ... = required
    capacity: int = Field(..., gt=0, le=10000)   # 0 < capacity <= 10000
    has_icu:  bool = False
    lat: float = Field(..., ge=-90, le=90)
    lon: float = Field(..., ge=-180, le=180)
    city: Optional[str] = None

## ۱۴-۴) اتصال به پایگاه‌داده با تزریق وابستگی


In [ ]:
from sqlalchemy.orm import sessionmaker
from database import engine          # the engine from Chapter 13
SessionLocal = sessionmaker(bind=engine)
def get_db():
    db = SessionLocal()
    try:
        yield db                      # hand the connection to the endpoint
    finally:
        db.close()                    # ... and always close it afterwards

## ۱۴-۵) نقطه‌ی اوج: مدل سیل پشت یک endpoint


In [ ]:
import joblib, numpy as np
from fastapi import APIRouter, Query

router = APIRouter(prefix="/predict", tags=["flood"])

# load ONCE at startup, not on every request
model = joblib.load("rf_flood.pkl")

@router.get("/flood-risk")           # a read-only prediction -> GET
def predict_flood_risk(
    slope: float = Query(..., ge=0, le=90),          # degrees
    ndvi: float = Query(..., ge=-1, le=1),
    dist_river_m: float = Query(..., ge=0),          # metres to a river
    elevation: float = Query(..., ge=-500, le=9000), # metres a.s.l.
):
    """Return the flood-risk probability for one location."""
    features = np.array([[slope, ndvi, dist_river_m, elevation]])
    prob = float(model.predict_proba(features)[0][1])   # P(flood)
    level = ("high" if prob > 0.66
             else "medium" if prob > 0.33
             else "low")
    return {"flood_probability": round(prob, 3), "risk_level": level}

# call it straight from the browser:
#   http://127.0.0.1:8000/predict/flood-risk
#       ?slope=3.4&ndvi=0.21&dist_river_m=180&elevation=1180

In [ ]:
from fastapi import APIRouter, Depends, Query
from sqlalchemy.orm import Session
from sqlalchemy import text
from dependencies import get_db

router = APIRouter(prefix="/analysis", tags=["analysis"])

@router.get("/nearest-hospitals")     # a read-only search -> GET
def nearest_hospitals(
    lat: float = Query(..., ge=-90, le=90),
    lon: float = Query(..., ge=-180, le=180),
    radius_km: float = Query(10, gt=0, le=100),
    db: Session = Depends(get_db),
):
    sql = text("""
        SELECT name, capacity,
               ST_Distance(geom::geography,
                   ST_SetSRID(ST_MakePoint(:lon, :lat), 4326)::geography
               ) / 1000 AS km
        FROM hospitals
        WHERE ST_DWithin(geom::geography,
                  ST_SetSRID(ST_MakePoint(:lon, :lat), 4326)::geography, :r)
        ORDER BY km
        LIMIT 10
    """)
    rows = db.execute(sql, {"lon": lon, "lat": lat,
                            "r": radius_km * 1000}).mappings().all()
    return {"count": len(rows), "results": [dict(r) for r in rows]}

## ۱۴-۶) خطاها، CORS و احراز هویت


In [ ]:
from fastapi import HTTPException, status

@app.get("/hospitals/{hid}")
def get_hospital(hid: int, db: Session = Depends(get_db)):
    row = db.execute(text("SELECT * FROM hospitals WHERE id = :id"),
                     {"id": hid}).fetchone()
    if row is None:
        raise HTTPException(status.HTTP_404_NOT_FOUND,
                            detail=f"hospital {hid} not found")
    return row._mapping

## ۱۴-۷) تست خودکار با pytest


In [ ]:
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

def test_root_ok():
    r = client.get("/")
    assert r.status_code == 200

def test_invalid_input_returns_422():
    r = client.get("/predict/flood-risk", params={"slope": 999})
    assert r.status_code == 422      # slope must be 0..90

# run the whole file with:  pytest -v